# Анализ закупок группы Сбер за 2024–2025 годы

В ноутбуке собран воспроизводимый аналитический отчёт по открытому контуру ЭТП.

- Юрлиц в периметре: `24`
- Юрлиц с наблюдаемыми закупками: `13`
- Лотов в итоговом слое после дедупликации: `1556`
- Удалено дублей: `65`
- Покрытие ценой: `25%`
- Сумма раскрытых цен: `2 498 248 140.68` RUB
- Строк с unit price: `2310`
- Извлечено документов: `250`
- Строк участников/продавцов: `616`
- Unit-price аномалий: `7`

## 1. Контур источников

Логика решения разделяет `официальный контур идентификации` и `рабочие контуры наблюдения`:

- `ЕИС` используется для резолвинга юридических лиц и контрольной проверки открытого покрытия по 223-ФЗ.
- `Росэлторг` и `Сбербанк-АСТ` используются как рабочие источники публичных карточек процедур.
- Для `Сбербанк-АСТ` дополнительно применяется фильтр по предметной области: из единого реестра исключены процедуры реализации имущества и банкротные продажи, чтобы в аналитический слой попадали только procurement-релевантные записи.
- Остальные ЭТП повторно исследованы и отражены в `source_assessment.csv` как `operational`, `research_only` или `blocked`.

Это важно, потому что задача не только про сбор данных, но и про честную оценку покрытия.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
CURATED = ROOT / 'data' / 'curated'

ENTITY_ID_COLUMNS = ['inn', 'resolved_inn', 'eis_resolved_inn', 'eis_resolved_kpp', 'eis_resolved_ogrn']
SOURCE_LINK_ID_COLUMNS = ['external_customer_key', 'external_inn', 'external_kpp']

def read_csv_with_ids(path, id_columns=()):
    df = pd.read_csv(path, dtype={column: 'string' for column in id_columns})
    for column in id_columns:
        if column in df.columns:
            df[column] = df[column].fillna('')
    return df

entities = read_csv_with_ids(CURATED / 'entity_coverage.csv', ENTITY_ID_COLUMNS)
source_assessment = pd.read_csv(CURATED / 'source_assessment.csv')
entity_links = read_csv_with_ids(CURATED / 'entity_source_links.csv', SOURCE_LINK_ID_COLUMNS)
lots = pd.read_csv(CURATED / 'procurement_lots.csv')
yearly = pd.read_csv(CURATED / 'mart_yearly_summary.csv')
monthly = pd.read_csv(CURATED / 'mart_monthly_activity.csv')
category_mix = pd.read_csv(CURATED / 'mart_category_mix.csv')
category_yoy = pd.read_csv(CURATED / 'mart_category_yoy.csv')
anomalies = pd.read_csv(CURATED / 'mart_anomalies.csv')
duplicate_stats = pd.read_csv(CURATED / 'duplicate_stats.csv')
macro = pd.read_csv(CURATED / 'mart_monthly_macro_join.csv')
macro_diagnostics = pd.read_csv(CURATED / 'mart_macro_diagnostics.csv')
items = pd.read_csv(CURATED / 'procurement_items.csv')
document_texts = pd.read_csv(CURATED / 'document_texts.csv')
participants = pd.read_csv(CURATED / 'procurement_participants.csv')
unit_price_benchmarks = pd.read_csv(CURATED / 'mart_unit_price_benchmarks.csv')

source_assessment[['source_system', 'operational_status', 'inclusion_status', 'coverage_note']]

## 2. Связка сущностей между источниками

Ниже показано, как юрлица группы Сбер были связаны с внешними идентификаторами источников.

Observation: для части компаний исходный ИНН в первоначальном scope пришлось скорректировать по ЕИС/AST-резолвингу.

Interpretation: без отдельного шага entity resolution мы получили бы ложные пропуски и слабое покрытие по закупкам.

Significance: это ключевой инженерный шаг для воспроизводимости и качества выгрузки.

Limitation: открытые справочники ЭТП показывают не все внутренние дочерние/региональные сущности одинаково полно.

In [ ]:
display(entities[['entity_name', 'inn', 'resolved_inn', 'eis_223_open_count', 'roseltorg_lot_count', 'sberbank_ast_lot_count']])
display(entity_links[['entity_name', 'source_system', 'external_customer_key', 'records_total']].head(40))

## 3. Сравнение 2024 vs 2025

Ниже — базовое сравнение числа лотов и раскрытого стоимостного объёма по годам.

In [ ]:
display(yearly)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
yearly.plot(x='publication_year', y='lots_count', kind='bar', ax=axes[0], legend=False, color='#1D3557')
axes[0].set_title('Количество лотов по годам')
axes[0].set_xlabel('Год')
axes[0].set_ylabel('Лоты')
yearly.plot(x='publication_year', y='total_price_rub', kind='bar', ax=axes[1], legend=False, color='#2A9D8F')
axes[1].set_title('Раскрытая стоимость по годам')
axes[1].set_xlabel('Год')
axes[1].set_ylabel('RUB')
plt.tight_layout()
plt.show()

Observation: в 2025 году открытый procurement-контур заметно активнее и по количеству лотов, и по раскрытой стоимости.

Interpretation: часть эффекта связана с более широким публичным покрытием AST в 2025 году и активизацией коммерческих закупок в цифровом/операционном контуре.

Significance: даже без полного внутреннего доступа можно уверенно зафиксировать межгодовой сдвиг активности.

Limitation: это не полная группа Сбер и не полный closed-loop procurement, а только наблюдаемый публичный слой.

## 4. Ключевое направление: Telecom & Devices

В качестве ключевого направления выбрано `Telecom & Devices`, потому что оно одновременно:

- хорошо наблюдается в открытых данных,
- содержит заметный стоимостной объём,
- даёт предметные закупочные сюжеты, а не только общие процедуры.

In [ ]:
telecom_yoy = category_yoy[category_yoy['focus_category'] == 'Telecom & Devices'].copy()
display(telecom_yoy)
monthly_pivot = monthly.pivot_table(index='publication_month', columns='focus_category', values='lots_count', fill_value=0)
monthly_pivot[['Telecom & Devices']].plot(figsize=(12, 4), marker='o', color='#E76F51')
plt.title('Telecom & Devices: помесячная динамика лотов')
plt.xlabel('Месяц')
plt.ylabel('Лоты')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Observation: направление `Telecom & Devices` выросло по количеству лотов примерно в `2.60x` между 2024 и 2025 годами.

Interpretation: это согласуется с публично наблюдаемыми закупками оборудования, лицензий, сетевой инфраструктуры и профильных ИТ-услуг.

Significance: направление можно использовать как приоритетное окно для детального мониторинга технологических потребностей группы.

Limitation: часть общих ИТ/операционных закупок остаётся в категории `Other`, потому что названия процедур не всегда достаточно структурированы.

## 5. Топ дорогих лотов

Здесь рассматриваются все закупочные лоты после фильтрации out-of-scope процедур: товары, работы и услуги. В 44-ФЗ и 223-ФЗ оказание услуг также относится к закупкам, поэтому строки вида `Оказание услуг...` корректно попадают в общий рейтинг. Этот блок нужен для отбора крупных процедур и ручной проверки, а не для вывода о завышенной цене за единицу. Для проверки завышения цен ниже используется отдельный контур unit-price benchmarks по сопоставимым позициям с единицами измерения.

In [ ]:
top_lots = (
    lots[['entity_name', 'platform_section', 'subject', 'focus_category', 'price_rub', 'published_at', 'detail_url']]
    .sort_values('price_rub', ascending=False)
    .head(20)
)
display(top_lots)
top_lots.head(10).plot(kind='barh', x='subject', y='price_rub', figsize=(12, 8), color='#457B9D', legend=False)
plt.title('Топ-10 дорогих лотов')
plt.xlabel('RUB')
plt.ylabel('Предмет')
plt.tight_layout()
plt.show()

Observation: верхние строки формируются в основном за счёт крупных телеком- и инфраструктурных закупок, особенно в контуре `ООО Сбербанк-Телеком`.

Interpretation: открытый публичный след особенно хорошо отражает крупные конкурентные процедуры с существенной закупочной стоимостью.

Significance: этот блок уже подходит для short-list ручной проверки, контроля категорий и последующей supplier-аналитики.

Limitation: по части процедур цена всё ещё не раскрывается, поэтому top-value лист не эквивалентен полной картине расходов.

## 6. Корреляция с макрофакторами

Проверяем исследовательскую гипотезу: меняется ли открытая закупочная активность вместе с динамикой `USD/RUB` и ключевой ставки.

In [ ]:
display(macro)
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(macro['publication_month'], macro['lots_count'], color='#C1121F', marker='o', label='Лоты')
ax1.set_ylabel('Лоты')
ax1.set_xlabel('Месяц')
ax1.tick_params(axis='x', rotation=45)
ax2 = ax1.twinx()
ax2.plot(macro['publication_month'], macro['avg_usd_rub'], color='#003049', marker='s', label='USD/RUB')
ax2.plot(macro['publication_month'], macro['avg_key_rate'], color='#669BBC', marker='^', label='Ключевая ставка')
if 'inflation_yoy_pct' in macro:
    ax2.plot(macro['publication_month'], macro['inflation_yoy_pct'], color='#2A9D8F', marker='x', label='ИПЦ г/г')
ax2.set_ylabel('Макрофакторы')
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')
plt.title('Открытая закупочная активность и макрофакторы')
plt.tight_layout()
plt.show()
display(macro_diagnostics)

Observation: текущий open-data sample даёт отрицательную корреляцию `lots_count vs USD` на уровне `-0.503` и отрицательную корреляцию `total_price vs key_rate` на уровне `0.236`.

Interpretation: это больше похоже на исследовательский сигнал, чем на устойчивую причинную связь.

Significance: архитектура уже позволяет регулярно проверять подобные гипотезы и расширять набор внешних факторов.

Limitation: выборка короткая и частично смещена в сторону отдельных источников, поэтому статистическую силу здесь нельзя переоценивать.

## 7. Дедупликация, документы и ПДн

Отдельно фиксируем служебные аспекты качества данных: дубли, документы, извлечение текста и обезличивание.

In [ ]:
display(duplicate_stats)
docs = pd.read_csv(CURATED / 'document_links.csv')
display(docs.head(20))
display(document_texts[['procedure_number', 'document_name', 'extraction_method', 'text_chars', 'ocr_required', 'pii_findings_count']].head(30))
document_texts[['text_chars', 'pii_findings_count']].describe()

Observation: документы SberB2B скачиваются ограниченным лимитом, текст извлекается из DOCX/PDF, а email/телефоны и похожие идентификаторы маскируются до попадания в витрину.

Interpretation: это превращает вложения из формального списка файлов в источник требований, номенклатуры и признаков цены.

Significance: блок закрывает ожидаемый в ТЗ навык проверки скачанных данных, обезличивания и подготовки данных для LLM/OCR-контура.

Limitation: если PDF почти не содержит текстового слоя, строка помечается `ocr_required=True`; OCR вынесен как следующий контролируемый шаг, чтобы не смешивать уверенный текст с распознаванием сомнительного качества.

## 8. Участники и продавцы

In [ ]:
display(participants.head(50))
participants.groupby(['source_system', 'participant_role', 'is_winner'], dropna=False).size().reset_index(name='rows')

Observation: Roseltorg detail cards expose `seller` from public JSON-LD, while public SberB2B cards in tested access mode do not expose winner/offer lists.

Interpretation: участники вынесены в отдельную таблицу с evidence_source, чтобы не смешивать продавца из карточки и фактического победителя.

Significance: это честный participant-level слой: он показывает, что извлечено, и явно фиксирует отсутствие победителей в публичном контуре.

Limitation: `winners_total=0` означает не провал парсинга, а отсутствие подтверждённого публичного winner endpoint без авторизации.

## 9. Unit-price benchmarks

In [ ]:
unit_flags = unit_price_benchmarks[unit_price_benchmarks['unit_price_anomaly_flag'] == True]
display(unit_flags.head(50))
unit_price_benchmarks['observations'].describe()

Observation: SberB2B goods API даёт строковые позиции с OKPD2, количеством, единицей измерения и unit price.

Interpretation: это позволяет перейти от анализа дорогих лотов к сравнению типовых товаров: бумага, мебель, спортинвентарь, печать, бытовая техника.

Significance: именно здесь появляется максимальная бизнес-ценность тестового — поиск завышенных цен по сопоставимым позициям, а не только красивый dashboard.

Limitation: benchmark требует достаточного числа наблюдений и нормализации наименований; поэтому флаги являются shortlist для ручной проверки, а не автоматическим обвинением.

## 10. Аномалии

In [ ]:
display(anomalies.head(50))
anomalies['anomaly_type'].value_counts()

Observation: доминирующий тип аномалий — `price_outlier`, что естественно для публичного слоя с заметным перекосом в крупные и редкие процедуры.

Interpretation: самые крупные телеком- и инфраструктурные закупки выбиваются относительно медианы своих категорий на порядки.

Significance: это уже полезный short-list для ручной проверки, аудита категории и сценариев anti-fraud/anti-waste.

Limitation: крупный лот не равен завышенной цене; более строгий контур находится в unit-price benchmarks и документах.

## 11. LLM-автоматизация

В репозитории дополнительно генерируется `data/reports/llm_prompt_pack.md` — компактный контекст-пакет для LLM.

Практический смысл:

- можно быстро отдавать витрины модели для генерации первичных выводов,
- можно автоматизировать drafting аналитической записки,
- можно использовать этот же пакет вместе с извлечёнными документами как основу для document/question-answer контура.

Это не заменяет ручную проверку, но ускоряет интерпретацию и подготовку narrative-части.

## Вывод

После повторного исследования ЭТП и расширения покрытия решение перестало быть одноисточниковым: теперь оно сочетает `ЕИС` для официального контроля, `Росэлторг` для лот-карточек и `Сбербанк-АСТ` для массового публичного procurement-контента. Ключевой инженерный результат — не просто рост объёма данных, а появление корректного, дедуплицированного и предметно очищенного аналитического слоя, который уже можно защищать как тестовое решение уровня Data Engineer / Data Analyst.